In [1]:
import pandas as pd
import sys
import os
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch 
from milvus import default_server
from pymilvus import FieldSchema, CollectionSchema, DataType, Collection, utility, connections

In [3]:
connections.connect("default", host="localhost", port="19530")

if connections.has_connection("default"):
    print("Successfully connected to Milvus")
else:
    print("Failed to connect to Milvus")


Successfully connected to Milvus


In [4]:
# Define the schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=50, is_primary=True),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=32768), 
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535)
]

schema = CollectionSchema(fields,  enable_dynamic_field=False)
col_name = 'splade_experiment'
col = Collection(col_name, schema, consistency_level="Strong")


In [5]:
index_params = {
    "metric_type": "COSINE",
    "index_type": "IVF_FLAT",
    "params": {"nlist": 1024}
}

col.create_index(field_name="embedding", index_params=index_params)
col.load()

In [6]:
tokenizer = AutoTokenizer.from_pretrained("naver/splade-cocondenser-ensembledistil")
model = AutoModelForMaskedLM.from_pretrained("naver/splade-cocondenser-ensembledistil")

/home/sbasir/Thesis/myenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/sbasir/Thesis/myenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
def merge_text_fields(data):
    for item in data:
        # Merge the fields into 'text', separating them by " | "
        merged_text = " | ".join(filter(None, [item.get('text', ''), 
                                            item.get('provided_data', ''), 
                                            item.get('enriched_data', ''), 
                                            item.get('translated_data', '')]))
        
        # Assign the merged text back to the 'text' field
        item['text'] = merged_text
        
        # Remove the individual fields as they're now part of 'text'
        item.pop('provided_data', None)
        item.pop('enriched_data', None)
        item.pop('translated_data', None)
    
    return data

def get_max_logits(output, tokens):
    return torch.max(
        torch.log(
            1 + torch.relu(output.logits)
        ) * tokens.attention_mask.unsqueeze(-1),
        dim=1)[0].squeeze().detach().cpu().numpy()

def builder(records: list):
    ids = [x['id'] for x in records]
    text = [x['text'] for x in records]
    # create sparse vecs
    tokens = tokenizer(
        text, return_tensors='pt',
        padding=True, truncation=True
    )
    sparse_vecs = get_max_logits(model(**tokens), tokens)
    upserts = []
    for _id, sparse_vec, text in zip(ids, sparse_vecs, text):
        upserts.append({
            'id': _id,
            'sparse_values': sparse_vec,
            'text': text
        })
    return upserts

def splade_embeddings(batch_data):
    data = merge_text_fields(batch_data)
    print(data[0])
    # Load the model
    upserts = builder(data)

    ids = [x['id'] for x in upserts]
    sparse_embeddings = [x['sparse_values'] for x in upserts]
    text = [x['text'] for x in upserts]

    entities = [
        {"id": _id, "embedding": _embedding, "text": _text}
        for _id, _embedding, _text in zip(ids, sparse_embeddings, text)
    ]

    return entities

In [8]:
import os
import gzip
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
import tqdm 
import sys

# Function to load data from a compressed JSON file
def load_compressed_json(file_path):
    """Function to load data from a compressed JSON file."""
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        return json.load(f)

# Function to load data in 64MB batches for Milvus indexing
def load_data_in_batches_for_indexing(parsed_directory, max_batch_size_mb=64, max_workers=4):
    """Load data from compressed JSON files in batches for indexing into Milvus."""
    max_batch_size_bytes = max_batch_size_mb * 1024 * 1024  # Convert MB to bytes
    batch_data = []  # To store the current batch
    batch_size = 0  # To track the size of the current batch in bytes

    # Collect all the .json.gz file paths
    file_paths = []
    for root, dirs, files in os.walk(parsed_directory):
        for file in files:
            if file.endswith('.json.gz'):
                file_path = os.path.join(root, file)
                file_paths.append(file_path)

    # Use ThreadPoolExecutor to load files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_file = {executor.submit(load_compressed_json, file_path): file_path for file_path in file_paths}

        for future in tqdm.tqdm(as_completed(future_to_file), total=len(future_to_file), desc="Loading and batching files"):
            file_path = future_to_file[future]
            try:
                data = future.result()  # Load the file's data
                data_size = sys.getsizeof(json.dumps(data))  # Get the size of this data in bytes

                # If the current batch size plus new data exceeds the limit, process the batch
                if (batch_size + data_size) > max_batch_size_bytes:
                    # Call your Milvus insertion function here
                    index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

                    # Reset the batch
                    batch_data = []
                    batch_size = 0

                # Add the data to the current batch
                batch_data.extend(data)  # Assuming data is a list of documents
                batch_size += data_size

            except Exception as e:
                print(f"Error loading file {file_path}: {e}")

    # Process any remaining data in the last batch
    if batch_data:
        index_batch_into_milvus(batch_data)  # Replace this with your Milvus indexing function

def index_batch_into_milvus(batch_data):
    print(f"Indexing batch with {len(batch_data)} documents into Milvus...")

    entities = splade_embeddings(batch_data)

    # Verify the lengths of each component to ensure they match
    print(f"Length of IDs: {len(entities[0])}")
    print(f"Length of texts: {len(entities[1])}")
    print(f"Shape of dense vectors: {len(entities[2])}")

    col.insert(entities)
    col.flush()

In [9]:
# run the function to load data in batches
load_data_in_batches_for_indexing('/home/sbasir/Thesis/Thesis/cp', max_batch_size_mb=64, max_workers=4)

Loading and batching files: 100%|██████████| 3/3 [00:00<00:00, 496.90it/s]

Indexing batch with 835 documents into Milvus...
{'id': '/5/URN_NBN_SI_IMG_APJFYR3F', 'text': "timestamp_update is 2019-07-08T19:33:17.185Z | type is IMAGE | content_tier is 0 | metadata_tier is A | edm:dataProvider is National and University Library, Ljubljana | National and University Library of Slovenia | edm:provider is Slovenski nacionalni agregator e-vsebin | Slovenian National E-content Aggregator | dc:description is Rokopis ni Kopitarjev, temveč je prepis Kopitarjeve razprave o jeziku današnjih štajerskih Slovencev po najstarejših staroslovanskih tekstih. Pisava po 24 vrstic&nbsp;na strani, žig ljubljanske licejske knjižnice pa je na ff. 1 in 32'.Cod. Kop. 27 | dc:format is 64 str. (32 f.) | dc:language is lat | dc:source is Kopitarjeva zbirka slovanskih kodeksov | dc:subject is rokopisi | dc:title is Kopitarjeva razprava o jeziku štajerskih Slovencev v razmerju do Clozovega zbornika | dc:type is rokopisi | dcterms:issued is 1836 | dc:source is Kopitariar Collection of Slavic C

: 

In [ ]:
query = input("Enter a query: ")

query_tokens = tokenizer(query, return_tensors="pt")
query_output = model(**query_tokens)

query_sparse_emb = get_max_logits(query_output, query_tokens)

search_params = {"metric_type": "COSINE", "params": {"nprobe": 10}}

results = collection.search(
    data=[query_sparse_emb],
    anns_field="embedding",
    param=search_params,
    limit=10,
    output_fields=["id", "text"],
)

print("query: ", query)
for result in results[0]:
    print(f"Document ID: {result.id}, Text: {result.entity.get('text')}, Distance: {result.distance}")

